In [ ]:
# Build ONE wide CSV for ONE run: TokenID, FullText, Target, FEAT_annotation/confidence/explanation...

from pathlib import Path
import pandas as pd
import re

# === SET THIS ===
run_dir = Path(r"/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/p1_cri_pipeline/runs/MODEL/FOLDER")  # <-- change me

# Collect candidate CSVs (exclude any existing master outputs)
all_csvs = [
    p for p in run_dir.rglob("*.csv")
    if "master" not in p.name.lower()
]

if not all_csvs:
    raise FileNotFoundError(f"No CSV files found under: {run_dir}")

def infer_feature_id(fname: str):
    # expects FEATID_...
    m = re.match(r"^([A-Za-z0-9]+)_", fname)
    return m.group(1) if m else None

# Group CSVs by feature_id
by_feat = {}
for p in all_csvs:
    feat = infer_feature_id(p.name)
    if feat is None:
        continue
    by_feat.setdefault(feat, []).append(p)

if not by_feat:
    raise RuntimeError(
        "Couldn't infer any feature IDs from filenames. "
        "Expected names like FEATID_FIRSTTOKEN_LASTTOKEN_YYYYMMDD.csv"
    )

# Choose one CSV per feature (policy: most recently modified)
chosen = {}
for feat, paths in by_feat.items():
    chosen[feat] = max(paths, key=lambda x: x.stat().st_mtime)

# Build base dataframe from one chosen file (keeps TokenID, FullText, Target)
# (Any feature file should contain these 3 because it’s the merged output.)
first_feat = sorted(chosen.keys())[0]
base_df = pd.read_csv(chosen[first_feat])

needed_base = ["TokenID", "FullText", "Target"]
missing_base = [c for c in needed_base if c not in base_df.columns]
if missing_base:
    raise RuntimeError(
        f"Base columns missing from {chosen[first_feat].name}: {missing_base}"
    )

wide = base_df[needed_base].copy()

# Merge each feature’s results into wide table
for feat in sorted(chosen.keys()):
    p = chosen[feat]
    df = pd.read_csv(p)

    # Require the prediction columns
    req = ["TokenID", "annotation", "confidence", "explanation"]
    miss = [c for c in req if c not in df.columns]
    if miss:
        raise RuntimeError(f"{p.name} is missing columns: {miss}")

    sub = df[req].copy()

    # Rename prediction columns with feature prefix
    sub = sub.rename(columns={
        "annotation": f"{feat}_annotation",
        "confidence": f"{feat}_confidence",
        "explanation": f"{feat}_explanation",
    })

    # Left merge on TokenID (keeps all tokens even if something failed)
    wide = wide.merge(sub, on="TokenID", how="left")

# Optional sanity check: expected columns count = 3 + 3*#features
print(f"Features merged: {len(chosen)}")
print(f"Rows: {len(wide)}")
print(f"Columns: {len(wide.columns)} (expected {3 + 3*len(chosen)})")

# Write output
run_id = run_dir.name
out_path = run_dir / f"{run_id}_WIDE.csv"
wide.to_csv(out_path, index=False)

print("✅ Wrote wide master CSV:")
print(out_path)